In [13]:
import os
import json
import cv2
import xml.etree.ElementTree as ET
import numpy as np
import pandas as pd
from tqdm import tqdm
from pathlib import Path
from matplotlib import pyplot as plt

In [16]:
ROOT = r"C:\Users\kiran\Downloads\Military-Camouflage-MHCD2022\Military-Camouflage-MHCD2022"

Img_Path = os.path.join(ROOT, "JPEGImages")
Anno_Path = os.path.join(ROOT, "Annotations")

Split_Path = os.path.join(ROOT, "ImageSets", "Main")

Output_Dir = r"C:\Users\kiran\Downloads\MHCD2022_COCO_Conversion"
os.makedirs(Output_Dir, exist_ok=True)



In [20]:
Class_MAP = {    
    "person" : 0,
    "military vehicle": 1,
    "tank" : 2,
    "aeroplane": 3,
    "warship" : 4 
}

In [26]:
def load_split(split_name):

    path = os.path.join(Split_Path, f"{split_name}.txt")
    
    with open(path, "r") as f:
        ids = [
            line.strip()
            for line in f.readlines()
            if line.strip()
        ]
        
    return ids

In [28]:
def parse_xml(xml_path):
    tree = ET.parse(xml_path)
    root = tree.getroot()

    width = int(root.find("size/width").text)
    height = int(root.find("size/height").text)

    objects = []
    for obj in root.findall("object"):
        cls = obj.find("name").text
        bbox = obj.find("bndbox")
        xmin = int(bbox.find("xmin").text)
        ymin = int(bbox.find("ymin").text)
        xmax = int(bbox.find("xmax").text)
        ymax = int(bbox.find("ymax").text)

        objects.append({
            "class": cls,
            "bbox": [xmin, ymin, xmax - xmin, ymax - ymin]
        })
    return width, height, objects

In [36]:
def create_coco(split_ids):

    coco = {
       "images" : [],
       "annotations" : [],
       "categories" : []
 }    
    for cls, cid in Class_MAP.items():
        coco["categories"].append({
            "id": cid,
            "name": cls
        })

    image_id =0
    ann_id =0

    for stem in tqdm(split_ids):
        xml_path = os.path.join(Anno_Path, stem + ".xml")
        img_path = os.path.join(Img_Path, stem + ".jpg")

        if not os.path.exists(xml_path) :
            continue
        if not os.path.exists(img_path) :
            continue
        width, height, objects = parse_xml(xml_path)

        coco["images"].append({
            "id": image_id,
            "file_name": stem + ".jpg",
            "width": width,
            "height": height
        })

        for obj in objects:

            if obj["class"] not in Class_MAP:
                continue

            x,y,w,h = obj["bbox"]
            coco["annotations"].append({
                "id": ann_id,
                "image_id": image_id,
                "category_id": Class_MAP[obj["class"]],
                "bbox": [int(x), int(y), int(w), int(h)],
                "area": w*h,
                "iscrowd": 0
            })
            
            ann_id += 1
        image_id += 1
    
    return coco   

In [37]:
train_ids = load_split("train")

train_coco = create_coco(train_ids)
with open(os.path.join(Output_Dir, "train_coco.json"), "w") as f:
    json.dump(train_coco, f, indent=4)  

100%|██████████| 2400/2400 [00:10<00:00, 233.85it/s]


In [38]:
test_ids = load_split("test")

test_coco = create_coco(test_ids)
with open(os.path.join(Output_Dir, "test_coco.json"), "w") as f:
    json.dump(test_coco, f, indent=4)

100%|██████████| 600/600 [00:02<00:00, 240.73it/s]


In [39]:
print("Train Images:", len(train_coco["images"]))
print("Train Annotations:", len(train_coco["annotations"]))
print("Average Objects/Image:", len(train_coco["annotations"])/len(train_coco["images"]))

Train Images: 2400
Train Annotations: 3481
Average Objects/Image: 1.4504166666666667
